# HumAID — Zero-shot Classification (Filtered Labels, Batch API, Sharding)

- **Filtered labels (per event):** prompts + JSON schema only list labels that appear in that event’s ground truth → reduces out-of-scope (OOS) predictions.
- **Batch API flow:** build `requests.jsonl` → upload → create batch → poll → download `outputs.jsonl` (and `errors.jsonl` if any).
- **Patch pass:** after batch completes, any missing/blank predictions are re-classified synchronously so `predictions.csv` has one row per input.
- **Stratified sharding (optional):** split large events into *k* shards **preserving class ratios**; use the **same** event-level labels + rules for all shards; merge predictions back in original order.
- **Reporting:** confusion matrices (counts + row-normalized), per-class F1/error, mistakes CSV, and a sortable `results/index.html`.  
  - **Scope** = label universe used for metrics (default `truth`).  
  - **OOS preds** = predictions not in the truth set (QA signal).

## Key settings
- `MODEL` (e.g., `gpt-4o`), `RULES` (e.g., `RULES_1`), `TAG`
- `DRYRUN_N`, `POLL_SECS`
- Token budgeting: `BATCH_TOKEN_LIMIT`, `SAFETY_MARGIN`, `MAX_OUTPUT_TOKENS`
- `.env` with `OPENAI_API_KEY_1` (and optionally a second key)

# 0) Setup

In [1]:
from pathlib import Path
import math
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment
from humaidclf import build_token_index               # token budgeting (sampling-based)
from humaidclf import run_experiment_sharded          # NEW: stratified sharded runner
from humaidclf.batch import use_api_key_env           # (optional) keep key switcher
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["test"]             # or ["train","dev","test"]
MODEL = "gpt-4.1"
RULES = RULES_1
TAG = "modeS-gpt-41-RULES1-filtered"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"

# Token caps & estimates
BATCH_TOKEN_LIMIT = 128_000   # Tier-1 cap
SAFETY_MARGIN = 0.90            # use only 90% of the cap
MAX_OUTPUT_TOKENS = 40          # matches your request schema

# 1) Discover datasets (events/splits)

In [2]:
def discover_tsvs(base: Path, splits: list[str]):
    items = []
    for event_dir in sorted([p for p in base.iterdir() if p.is_dir()]):
        event = event_dir.name
        for split in splits:
            tsv = event_dir / f"{event}_{split}.tsv"
            if tsv.exists():
                items.append({"event": event, "split": split, "tsv": str(tsv)})
    return pd.DataFrame(items)

df_sources = discover_tsvs(BASE, SPLITS)

# --- token budgeting ---
token_index = build_token_index(
    df_sources,
    model=MODEL,
    rules_text=RULES,
    batch_token_limit=BATCH_TOKEN_LIMIT,
    safety_margin=SAFETY_MARGIN,
    sample_size=200,
    max_output_tokens=MAX_OUTPUT_TOKENS,
)

display(token_index)

df_fit     = token_index[token_index["fits_cap"]].reset_index(drop=True)
df_too_big = token_index[~token_index["fits_cap"]].reset_index(drop=True)

print("OK to run as single batch:")
display(df_fit[["event","split","num_rows","est_total_tokens","limit_used_%"]])

print("Will be sharded (exceeds cap):")
display(df_too_big[["event","split","num_rows","est_total_tokens","limit_used_%"]])

,event,split,tsv,num_rows,avg_req_tokens,est_total_tokens,fits_cap,limit_used_%
1,canada_wildfires_2016,test,Dataset\HumAID\canada_wildfires_2016\canada_wi...,445,473,210485,False,164.4
8,kaikoura_earthquake_2016,test,Dataset\HumAID\kaikoura_earthquake_2016\kaikou...,435,487,211845,False,165.5
2,cyclone_idai_2019,test,Dataset\HumAID\cyclone_idai_2019\cyclone_idai_...,779,519,404301,False,315.9
4,hurricane_florence_2018,test,Dataset\HumAID\hurricane_florence_2018\hurrica...,1241,501,621741,False,485.7
7,hurricane_maria_2017,test,Dataset\HumAID\hurricane_maria_2017\hurricane_...,1442,487,702254,False,548.6
0,california_wildfires_2018,test,Dataset\HumAID\california_wildfires_2018\calif...,1461,507,740727,False,578.7
3,hurricane_dorian_2019,test,Dataset\HumAID\hurricane_dorian_2019\hurricane...,1508,503,758524,False,592.6
9,kerala_floods_2018,test,Dataset\HumAID\kerala_floods_2018\kerala_flood...,1582,508,803656,False,627.9
5,hurricane_harvey_2017,test,Dataset\HumAID\hurricane_harvey_2017\hurricane...,1805,486,877230,False,685.3
6,hurricane_irma_2017,test,Dataset\HumAID\hurricane_irma_2017\hurricane_i...,1862,486,904932,False,707.0


OK to run as single batch:


,event,split,num_rows,est_total_tokens,limit_used_%


Will be sharded (exceeds cap):


,event,split,num_rows,est_total_tokens,limit_used_%
0,canada_wildfires_2016,test,445,210485,164.4
1,kaikoura_earthquake_2016,test,435,211845,165.5
2,cyclone_idai_2019,test,779,404301,315.9
3,hurricane_florence_2018,test,1241,621741,485.7
4,hurricane_maria_2017,test,1442,702254,548.6
5,california_wildfires_2018,test,1461,740727,578.7
6,hurricane_dorian_2019,test,1508,758524,592.6
7,kerala_floods_2018,test,1582,803656,627.9
8,hurricane_harvey_2017,test,1805,877230,685.3
9,hurricane_irma_2017,test,1862,904932,707.0


# 2) Run all datasets (sequentially)

In [3]:
def run_list_single(dflist: pd.DataFrame, rules_text: str, model: str, tag: str):
    """Run events that already fit under the cap using the normal runner."""
    results = []
    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        print(f"\n=== Running (single) {event}/{split} ({model} | {tag}) ===")
        try:
            plan, preds, summary = run_experiment(
                dataset_path=tsv,
                rules=rules_text,
                model=model,
                tag=tag,
                dryrun_n=DRYRUN_N,
                poll_secs=POLL_SECS,
                out_root=OUT_ROOT,
                do_analysis=DO_ANALYSIS,
            )
            acc = summary.get("accuracy") if summary else float("nan")
            f1  = summary.get("macro_f1") if summary else float("nan")
            n   = summary.get("num_total_with_truth") if summary else len(preds)
            results.append({
                "event": event, "split": split,
                "run_dir": str(plan["dir"]),
                "predictions_csv": str(plan["predictions_csv"]),
                "macro_f1": f1, "accuracy": acc, "num_total": n,
                "mode": "single",
            })
        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")
            results.append({
                "event": event, "split": split, "run_dir": "ERROR",
                "predictions_csv": "", "macro_f1": float("nan"),
                "accuracy": float("nan"), "num_total": 0, "mode": "single",
            })
    return pd.DataFrame(results)

def run_list_sharded(dflist: pd.DataFrame, token_df: pd.DataFrame, rules_text: str, model: str, tag: str):
    """Run events that exceed the cap using stratified shards. k is computed from token estimates."""
    results = []
    # Build a quick lookup: (event,split) -> est_total_tokens
    est_map = {(r.event, r.split): r.est_total_tokens for r in token_df.itertuples(index=False)}
    eff_cap = BATCH_TOKEN_LIMIT * SAFETY_MARGIN

    for _, row in dflist.iterrows():
        event, split, tsv = row["event"], row["split"], row["tsv"]
        est_tokens = est_map.get((event, split), None)
        # Conservative shard count: ceil(est / eff_cap). Min 2.
        k = max(2, math.ceil((est_tokens or (eff_cap + 1)) / eff_cap))
        print(f"\n=== Running (sharded x{k}) {event}/{split} ({model} | {tag}) ===")
        try:
            plan, preds, summary = run_experiment_sharded(
                dataset_path=tsv,
                rules=rules_text,
                model=model,
                tag=f"{tag}-sharded{k}",
                k_shards=k,
                temperature=0.0,
                poll_secs=POLL_SECS,
                out_root=OUT_ROOT,
                do_analysis=DO_ANALYSIS,
                analysis_subdir="analysis",  # merged analysis
            )
            acc = summary.get("accuracy") if summary else float("nan")
            f1  = summary.get("macro_f1") if summary else float("nan")
            n   = summary.get("num_total_with_truth") if summary else len(preds)
            results.append({
                "event": event, "split": split,
                "run_dir": str(plan["dir"]),
                "predictions_csv": str(plan["predictions_csv"]),
                "macro_f1": f1, "accuracy": acc, "num_total": n,
                "mode": f"sharded{k}",
            })
        except Exception as e:
            print(f"[ERROR] {event}/{split}: {e}")
            results.append({
                "event": event, "split": split, "run_dir": "ERROR",
                "predictions_csv": "", "macro_f1": float("nan"),
                "accuracy": float("nan"), "num_total": 0, "mode": f"sharded{k}",
            })
    return pd.DataFrame(results)

In [4]:
# --- Run singles with your normal key (optional context manager kept for parity)
with use_api_key_env("OPENAI_API_KEY_1"):
    print(">>> Using OPENAI_API_KEY_1")
    df_runs_single = run_list_single(df_fit, RULES, MODEL, tag=f"{TAG}-TIER1")

# --- Run sharded for the too-big ones (same key or another if you prefer)
# You can keep the same key; sharding is already controlling token usage.
with use_api_key_env("OPENAI_API_KEY_1"):
    if not df_too_big.empty:
        df_runs_sharded = run_list_sharded(df_too_big, token_index, RULES, MODEL, tag=f"{TAG}")
    else:
        df_runs_sharded = pd.DataFrame()
        print("No large datasets to shard.")

# 3) Save a small index of all runs
from datetime import datetime
idx_dir = Path(OUT_ROOT) / "_indexes"
idx_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")

all_runs = pd.concat([df_runs_single, df_runs_sharded], ignore_index=True)
all_runs.to_csv(idx_dir / f"runs_{MODEL}_{TAG}_{stamp}.csv", index=False)
print("Saved run index at:", idx_dir)
display(all_runs)

>>> Using OPENAI_API_KEY_1

=== Running (sharded x2) canada_wildfires_2016/test (gpt-4.1 | modeS-gpt-41-RULES1-filtered) ===
[batch batch_690adbff3acc81909c9320523d331aa7] status = validating
[batch batch_690adbff3acc81909c9320523d331aa7] status = in_progress
[batch batch_690adbff3acc81909c9320523d331aa7] status = in_progress
[batch batch_690adbff3acc81909c9320523d331aa7] status = in_progress
[batch batch_690adbff3acc81909c9320523d331aa7] status = completed
[batch batch_690ae0b7da988190a2314ad9069ec53a] status = validating
[batch batch_690ae0b7da988190a2314ad9069ec53a] status = in_progress
[batch batch_690ae0b7da988190a2314ad9069ec53a] status = in_progress
[batch batch_690ae0b7da988190a2314ad9069ec53a] status = in_progress
[batch batch_690ae0b7da988190a2314ad9069ec53a] status = completed
Saved merged predictions to: runs\canada_wildfires_2016\test\gpt-4.1\20251104-210916-modeS-gpt-41-RULES1-filtered-sharded2\predictions.csv
Macro-F1 (merged): 0.6822047952159465

=== Running (sharded x2

,event,split,run_dir,predictions_csv,macro_f1,accuracy,num_total,mode
0,canada_wildfires_2016,test,runs\canada_wildfires_2016\test\gpt-4.1\202511...,runs\canada_wildfires_2016\test\gpt-4.1\202511...,0.682205,0.773034,445,sharded2
1,kaikoura_earthquake_2016,test,runs\kaikoura_earthquake_2016\test\gpt-4.1\202...,runs\kaikoura_earthquake_2016\test\gpt-4.1\202...,0.653432,0.673563,435,sharded2
2,cyclone_idai_2019,test,runs\cyclone_idai_2019\test\gpt-4.1\20251104-2...,runs\cyclone_idai_2019\test\gpt-4.1\20251104-2...,0.630455,0.717587,779,sharded4
3,hurricane_florence_2018,test,runs\hurricane_florence_2018\test\gpt-4.1\2025...,runs\hurricane_florence_2018\test\gpt-4.1\2025...,0.697514,0.734085,1241,sharded6
4,hurricane_maria_2017,test,runs\hurricane_maria_2017\test\gpt-4.1\2025110...,runs\hurricane_maria_2017\test\gpt-4.1\2025110...,0.623838,0.676144,1442,sharded7
5,california_wildfires_2018,test,runs\california_wildfires_2018\test\gpt-4.1\20...,runs\california_wildfires_2018\test\gpt-4.1\20...,0.635326,0.710472,1461,sharded7
6,hurricane_dorian_2019,test,runs\hurricane_dorian_2019\test\gpt-4.1\202511...,runs\hurricane_dorian_2019\test\gpt-4.1\202511...,0.560272,0.602785,1508,sharded7
7,kerala_floods_2018,test,ERROR,,NaN,NaN,0,sharded7
8,hurricane_harvey_2017,test,ERROR,,NaN,NaN,0,sharded8
9,hurricane_irma_2017,test,ERROR,,NaN,NaN,0,sharded8


# Other experiments

In [2]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment_sharded
from humaidclf.batch import use_api_key_env
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["test"]             # or ["train","dev","test"]
MODEL = "gpt-4.1"
RULES = RULES_1
TAG = "modeS-gpt-41-RULES1-filtered"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"
K = 10  # number of stratified shards

with use_api_key_env("OPENAI_API_KEY"):
    plan, preds, summary = run_experiment_sharded(
        dataset_path=str(BASE / "kerala_floods_2018" / "kerala_floods_2018_test.tsv"),
        rules=RULES,
        model=MODEL,
        tag=f"{TAG}-sharded{K}",
        k_shards=K,
        temperature=0.0,
        poll_secs=POLL_SECS,
        out_root=OUT_ROOT,
        do_analysis=DO_ANALYSIS,
        analysis_subdir="analysis",
    )

summary



[batch batch_690ba1ceead08190a214e4c4d1bbfa73] status = validating
[batch batch_690ba1ceead08190a214e4c4d1bbfa73] status = completed
[batch batch_690ba2feb6508190a9928b83255f351c] status = validating
[batch batch_690ba2feb6508190a9928b83255f351c] status = completed
[batch batch_690ba42ecbd48190afc95659419329fb] status = validating
[batch batch_690ba42ecbd48190afc95659419329fb] status = completed
[batch batch_690ba55d68408190a3cb2cf641bc1c98] status = validating
[batch batch_690ba55d68408190a3cb2cf641bc1c98] status = completed
[batch batch_690ba68e41388190b0bdd6e9753a3571] status = validating
[batch batch_690ba68e41388190b0bdd6e9753a3571] status = completed
[batch batch_690ba7bd79ac81908925acf54c4ec3ab] status = validating
[batch batch_690ba7bd79ac81908925acf54c4ec3ab] status = completed
[batch batch_690ba8ed444c81908493838be73b6643] status = validating
[batch batch_690ba8ed444c81908493838be73b6643] status = completed
[batch batch_690baa1dc10481908a8365a90f306cb6] status = validating
[b

{'num_total_with_truth': 1582,
 'num_correct': 1064,
 'num_incorrect': 518,
 'accuracy': 0.672566371681416,
 'macro_f1': 0.5464655824450324,
 'labels': ['caution_and_advice',
  'displaced_people_and_evacuations',
  'infrastructure_and_utility_damage',
  'injured_or_dead_people',
  'not_humanitarian',
  'other_relevant_information',
  'requests_or_urgent_needs',
  'rescue_volunteering_or_donation_effort',
  'sympathy_and_support'],
 'labels_scope': 'truth',
 'invalid_pred_outside_truth': 0}

In [1]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment_sharded
from humaidclf.batch import use_api_key_env
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["test"]             # or ["train","dev","test"]
MODEL = "gpt-4.1"
RULES = RULES_1
TAG = "modeS-gpt-41-RULES1-filtered"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"
K = 10  # number of stratified shards

with use_api_key_env("OPENAI_API_KEY"):
    plan, preds, summary = run_experiment_sharded(
        dataset_path=str(BASE / "hurricane_harvey_2017" / "hurricane_harvey_2017_test.tsv"),
        rules=RULES,
        model=MODEL,
        tag=f"{TAG}-sharded{K}",
        k_shards=K,
        temperature=0.0,
        poll_secs=POLL_SECS,
        out_root=OUT_ROOT,
        do_analysis=DO_ANALYSIS,
        analysis_subdir="analysis",
    )

summary

[batch batch_690bc38c0214819081e436b2964d3c68] status = validating
[batch batch_690bc38c0214819081e436b2964d3c68] status = completed
[batch batch_690bc4bc4b2081908890c870f35d7c96] status = validating
[batch batch_690bc4bc4b2081908890c870f35d7c96] status = completed
[batch batch_690bc5eb89c881909da21346cfbcc161] status = validating
[batch batch_690bc5eb89c881909da21346cfbcc161] status = completed
[batch batch_690bc71b1be48190aa303dad0f4e94b4] status = validating
[batch batch_690bc71b1be48190aa303dad0f4e94b4] status = completed
[batch batch_690bc84a954c81908af685a937f720bc] status = validating
[batch batch_690bc84a954c81908af685a937f720bc] status = completed
[batch batch_690bc97981d88190a5c04ac62ecb56b5] status = validating
[batch batch_690bc97981d88190a5c04ac62ecb56b5] status = completed
[batch batch_690bcaa861e88190b9bbea36a037f320] status = validating
[batch batch_690bcaa861e88190b9bbea36a037f320] status = in_progress
[batch batch_690bcaa861e88190b9bbea36a037f320] status = in_progress

{'num_total_with_truth': 1805,
 'num_correct': 1204,
 'num_incorrect': 601,
 'accuracy': 0.6670360110803324,
 'macro_f1': 0.6201783848647233,
 'labels': ['caution_and_advice',
  'displaced_people_and_evacuations',
  'infrastructure_and_utility_damage',
  'injured_or_dead_people',
  'not_humanitarian',
  'other_relevant_information',
  'requests_or_urgent_needs',
  'rescue_volunteering_or_donation_effort',
  'sympathy_and_support'],
 'labels_scope': 'truth',
 'invalid_pred_outside_truth': 0}

In [1]:
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv; load_dotenv()

from humaidclf import run_experiment_sharded
from humaidclf.batch import use_api_key_env
from rules import RULES_1

# --- config ---
BASE = Path("Dataset/HumAID")
SPLITS = ["test"]             # or ["train","dev","test"]
MODEL = "gpt-4.1"
RULES = RULES_1
TAG = "modeS-gpt-41-RULES1-filtered"
DRYRUN_N = 20
POLL_SECS = 300
DO_ANALYSIS = True
OUT_ROOT = "runs"
K = 10  # number of stratified shards

with use_api_key_env("OPENAI_API_KEY"):
    plan, preds, summary = run_experiment_sharded(
        dataset_path=str(BASE / "hurricane_irma_2017" / "hurricane_irma_2017_test.tsv"),
        rules=RULES,
        model=MODEL,
        tag=f"{TAG}-sharded{K}",
        k_shards=K,
        temperature=0.0,
        poll_secs=POLL_SECS,
        out_root=OUT_ROOT,
        do_analysis=DO_ANALYSIS,
        analysis_subdir="analysis",
    )

summary

[batch batch_690c6fa56eb48190a6015b922344bf0a] status = validating
[batch batch_690c6fa56eb48190a6015b922344bf0a] status = completed
[batch batch_690c70d4f1308190b558c5c1b678aff0] status = validating
[batch batch_690c70d4f1308190b558c5c1b678aff0] status = completed
[batch batch_690c7204590c8190a7da4057d6fcaf63] status = validating
[batch batch_690c7204590c8190a7da4057d6fcaf63] status = completed
[batch batch_690c73331fbc819092c910c725d983f4] status = validating
[batch batch_690c73331fbc819092c910c725d983f4] status = in_progress
[batch batch_690c73331fbc819092c910c725d983f4] status = completed
[batch batch_690c758e9c048190aab8cfb756ca722b] status = validating
[batch batch_690c758e9c048190aab8cfb756ca722b] status = completed
[batch batch_690c76bd39bc819096123dcbd93c8121] status = validating
[batch batch_690c76bd39bc819096123dcbd93c8121] status = in_progress
[batch batch_690c76bd39bc819096123dcbd93c8121] status = in_progress
[batch batch_690c76bd39bc819096123dcbd93c8121] status = in_progr

{'num_total_with_truth': 1862,
 'num_correct': 1224,
 'num_incorrect': 638,
 'accuracy': 0.6573576799140709,
 'macro_f1': 0.6328123622462498,
 'labels': ['caution_and_advice',
  'displaced_people_and_evacuations',
  'infrastructure_and_utility_damage',
  'injured_or_dead_people',
  'not_humanitarian',
  'other_relevant_information',
  'requests_or_urgent_needs',
  'rescue_volunteering_or_donation_effort',
  'sympathy_and_support'],
 'labels_scope': 'truth',
 'invalid_pred_outside_truth': 0}